In [7]:
import pandas as pd
from pathlib import Path

ZILLOW_DIR = Path("../data/raw/zillow")

# 1. What files are actually there
print("=== FILES ===")
for f in sorted(ZILLOW_DIR.glob("*.csv")):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

# 2. Load ZHVI
# Load the ZIP-level file specifically
zhvi = pd.read_csv(ZILLOW_DIR / "Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

print(f"\n=== ZHVI SHAPE ===")
print(f"Rows: {len(zhvi):,}")
print(f"Columns: {len(zhvi.columns)}")

id_cols = [c for c in zhvi.columns if not c[0].isdigit()]
date_cols = [c for c in zhvi.columns if c[0].isdigit()]
print(f"\n=== COLUMNS ===")
print(f"ID columns ({len(id_cols)}): {id_cols}")
print(f"Date columns: {len(date_cols)}")
print(f"First date: {date_cols[0]}")
print(f"Last date: {date_cols[-1]}")

print(f"\n=== REGION TYPES ===")
print(zhvi["RegionType"].unique() if "RegionType" in zhvi.columns else "no RegionType column")

print(f"\n=== REGIONNAME DTYPE ===")
print(f"dtype: {zhvi['RegionName'].dtype}")
print(f"sample values: {zhvi['RegionName'].head(3).tolist()}")

print(f"\n=== SANTA CLARA ===")
if "CountyName" in zhvi.columns:
    sc = zhvi[zhvi["CountyName"].str.contains("Santa Clara", na=False)]
    print(f"Santa Clara rows: {len(sc)}")
    if len(sc) > 0:
        print(sc[["RegionName", "City", "CountyName"]].head(10).to_string())
else:
    print("no CountyName column — listing all columns:")
    print(id_cols)

print(f"\n=== MISSING DATA ===")
print(f"Oldest ({date_cols[0]}): {zhvi[date_cols[0]].isna().sum():,} / {len(zhvi):,} missing")
print(f"Newest ({date_cols[-1]}): {zhvi[date_cols[-1]].isna().sum():,} / {len(zhvi):,} missing")

print(f"\n=== FIRST 3 ROWS (ID cols only) ===")
print(zhvi[id_cols].head(3).to_string())

# 3. ID columns vs date columns
id_cols = [c for c in zhvi.columns if not c[0].isdigit()]
date_cols = [c for c in zhvi.columns if c[0].isdigit()]
print(f"\n=== COLUMNS ===")
print(f"ID columns ({len(id_cols)}): {id_cols}")
print(f"Date columns: {len(date_cols)}")
print(f"First date: {date_cols[0]}")
print(f"Last date: {date_cols[-1]}")

# 4. Region type check
print(f"\n=== REGION TYPES ===")
print(zhvi["RegionType"].unique() if "RegionType" in zhvi.columns else "no RegionType column")

# 5. RegionName dtype
print(f"\n=== REGIONNAME DTYPE ===")
print(f"dtype: {zhvi['RegionName'].dtype}")
print(f"sample values: {zhvi['RegionName'].head(3).tolist()}")

# 6. Santa Clara check
print(f"\n=== SANTA CLARA ===")
if "CountyName" in zhvi.columns:
    sc = zhvi[zhvi["CountyName"] == "Santa Clara County"]
    if len(sc) == 0:
        sc = zhvi[zhvi["CountyName"].str.contains("Santa Clara", na=False)]
    print(f"Santa Clara rows: {len(sc)}")
    if len(sc) > 0:
        print(sc[["RegionName", "City"]].head(5).to_string())
else:
    print("no CountyName column")

# 7. Missing data check
print(f"\n=== MISSING DATA ===")
print(f"Oldest ({date_cols[0]}): {zhvi[date_cols[0]].isna().sum():,} / {len(zhvi):,} missing")
print(f"Newest ({date_cols[-1]}): {zhvi[date_cols[-1]].isna().sum():,} / {len(zhvi):,} missing")

# 8. First few rows
print(f"\n=== FIRST 3 ROWS (ID cols only) ===")
print(zhvi[id_cols].head(3).to_string())

=== FILES ===
  Metro_zhvf_growth_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv  (0.1 MB)
  Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv  (4.4 MB)
  Metro_zori_uc_sfrcondomfr_sm_month.csv  (1.0 MB)
  Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv  (121.1 MB)
  Zip_zori_uc_sfrcondomfr_sm_month.csv  (9.3 MB)

=== ZHVI SHAPE ===
Rows: 26,283
Columns: 324

=== COLUMNS ===
ID columns (9): ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName']
Date columns: 315
First date: 2000-01-31
Last date: 2026-03-31

=== REGION TYPES ===
<ArrowStringArray>
['zip']
Length: 1, dtype: str

=== REGIONNAME DTYPE ===
dtype: int64
sample values: [77494, 8701, 77449]

=== SANTA CLARA ===
Santa Clara rows: 56
     RegionName         City          CountyName
92        95035     Milpitas  Santa Clara County
264       95123     San Jose  Santa Clara County
306       95020       Gilroy  Santa Clara County
392       95127     San Jose  Santa Clara County


In [ ]:
import sqlite3
from pathlib import Path

db_path = Path("../data/processed/housing.db")
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

# How many geographies by type?
print("=== GEOGRAPHIES ===")
rows = conn.execute("""
    SELECT geo_type, COUNT(*) as count 
    FROM geographies 
    GROUP BY geo_type 
    ORDER BY count DESC
""").fetchall()
for r in rows:
    print(f"  {r['geo_type']}: {r['count']:,}")

# How many metrics by type?
print("\n=== METRICS ===")
rows = conn.execute("""
    SELECT metric_type, COUNT(*) as count 
    FROM housing_metrics 
    GROUP BY metric_type
""").fetchall()
for r in rows:
    print(f"  {r['metric_type']}: {r['count']:,}")

# Santa Clara spot check — median home value over time
print("\n=== SANTA CLARA ZIP 95051 (last 5 dates) ===")
rows = conn.execute("""
    SELECT g.geo_code, g.name, h.metric_date, h.metric_type, h.value
    FROM housing_metrics h
    JOIN geographies g ON g.geography_id = h.geography_id
    WHERE g.geo_code = '95051' AND h.metric_type = 'zhvi'
    ORDER BY h.metric_date DESC
    LIMIT 5
""").fetchall()
for r in rows:
    print(f"  {r['metric_date']}: ${r['value']:,.0f}")

# Bay Area metro check
print("\n=== SAN JOSE METRO (last 5 dates) ===")
rows = conn.execute("""
    SELECT g.name, h.metric_date, h.value
    FROM housing_metrics h
    JOIN geographies g ON g.geography_id = h.geography_id
    WHERE g.geo_code LIKE '%San Jose%' AND g.geo_type = 'metro' AND h.metric_type = 'zhvi'
    ORDER BY h.metric_date DESC
    LIMIT 5
""").fetchall()
for r in rows:
    print(f"  {r['metric_date']}: ${r['value']:,.0f}")

conn.close()
